In [19]:
import json
import yaml
import io
import re
import os

topics = set()
subs = {}

STATE_INDEX = 1
PORT_INDEX = 1
def expr_to_PlusCal(str_):
    return str_.replace("/", "").replace(".", "_").replace("!", "~").replace("==", "=").replace("(anonymous namespace)::","").replace("::","_").replace("||","|").replace("&&","&")

def input_port_name(topic, component):
    return f'{topic}_{component}'


In [20]:
from sympy.parsing.sympy_parser import parse_expr
from sympy.logic import simplify_logic
from sympy.printing.pretty.pretty import pretty_print 
from sympy.printing.latex import LatexPrinter, print_latex
from sympy.printing.pretty.stringpict import stringPict
from sympy import *


In [21]:
def name_to_penrose(str_):
    return expr_to_PlusCal(str_).replace("_", "\_")

def expr_to_penrose(str_):
    str_= str_.replace("/", "").replace(".", "_").replace("!", "~").replace("=", "==").replace("(anonymous namespace)::","").replace("::","_").replace("||","|").replace("&&","&")
    str_ = str_.replace("_", "dash")
    expr = parse_expr(str_, evaluate=False)
    s = simplify_logic(expr, force=True, dontcare=False)
    x_eq_y_str = pretty(s,  use_unicode=False)
    expr = parse_expr(x_eq_y_str.replace("!= False","").replace("== True","").replace("=", "=="))
    s = simplify_logic(expr, force=True, dontcare=False)
    r = latex(s)
    r = r.replace("dash","\_")
    r = r.replace("\\text{True}","")
    #r = r.replace("_", "\_").replace("{", "").replace("}", "")
    #r = x_eq_y_str.replace("_", "\_").replace("True","") 
    if r:
        return "[" + r + "]"
    else:
        return " "
    

expr_to_penrose("(g_pose_set & g_waypoint_set) ")

'[g\\_pose\\_set \\wedge g\\_waypoint\\_set]'

In [22]:
expr_to_penrose("g_pose_set & g_waypoint_set")

'[g\\_pose\\_set \\wedge g\\_waypoint\\_set]'

In [23]:
expr = parse_expr("~((True | False) == gpose)", evaluate=False)
s = simplify_logic(expr)
x_eq_y_str = pretty(s,  use_unicode=True)


In [24]:

def outVariablesTLA(jsonComp):
    vars = "variables\n\t\tmsg \in Data;\n"
    
    for state_var in jsonComp["potential_state_vars"]:
        variable = expr_to_PlusCal(state_var["qualified_name"])
        initial_value = expr_to_PlusCal(str(state_var["initial-value"]["literal"]))
        vars += f"\t\t{variable} = {initial_value};\n" 
    return vars


def topicVariables():
    variables = ''
    invariants = []
    for topic in topics:
        variables += f'\t{topic} = <<>>;\n'
        invariants.append(f'\t{topic} \in SeqOf(Data, MaxQueue)')

        if topic not in subs:
            continue
        for sub in subs[topic]:
            variables += f'\t{sub} = <<>>;\n'
            invariants.append(f'\t{sub} \in SeqOf(Data, MaxQueue)')
    
    for config in configuration:
        variable = expr_to_PlusCal(config)
        initial_value = expr_to_PlusCal(configuration[config])
        variables += f"\t{variable} = {initial_value};\n" 
    invariants_txt =  " /\\ \n".join(invariants)
    return f"""variables
    {variables}
    define 
        TypeInvariant == \n {invariants_txt}

        Response == <>({desired_output} /= <<>>)
    end define;"""

def transitionsPensrose(jsonComp):
    global STATE_INDEX
    global PORT_INDEX
    reactive_behavior = {}
    comp_name = jsonComp["node"].capitalize()
    transitions = ''
    for rb in jsonComp["reactive_behavior"]:
        if "event" in rb:
            continue
        input_topic = expr_to_PlusCal(rb["subscriber"]["topic"])
        topics.add(input_topic)
        if input_topic not in subs:
            subs[input_topic] = set()
        inport = input_port_name(input_topic, comp_name)

        
        subs[input_topic].add(inport)

        transitionID = expr_to_PlusCal(rb["subscriber"]["callback"])
        if inport not in reactive_behavior:
            reactive_behavior[inport] = ""
        outport = expr_to_PlusCal(rb["publisher"]["topic"])
        reactive_behavior[inport] += f"{outport} := {outport} (+) msg;\n"
        if outport not in topics:
            topics.add(outport)
        
    for t in jsonComp["transitions"]:
        if t["type"] == "message":
            input_topic = expr_to_PlusCal(t["topic"])
            topics.add(input_topic)
            if input_topic not in subs:
                subs[input_topic] = set()
            inport = input_port_name(input_topic, comp_name)
            subs[input_topic].add(inport)
            transitionID = expr_to_PlusCal(t["callback"])
            condition = expr_to_PlusCal(t["condition"])

            outputs = ""
            for o in t["outputs"]:
                outport = expr_to_PlusCal(o["publisher"]["topic"])
                if outport not in topics:
                    topics.add(outport)
                    outputs += f"                            {outport} := {outport} (+) msg;"
            
            state_changes = []
            for sc in t["state_changes"]:
                variable  = name_to_penrose(sc["variable"])
                new_value =  expr_to_PlusCal(sc["new_value"])
                state_changes.append(f'{variable} := {new_value}')

            if inport not in reactive_behavior:
                reactive_behavior[inport] = ""
            if len(state_changes) > 1 or len(outputs) > 1:
                reactive_behavior[inport] += (f"""
                            if {condition} then
    {state_changes}
    {outputs}
                            end if;
                    """)
            port_name = f'{comp_name}_{input_topic}'
            state_changes_str = "\mathbf{" + " /\\ ".join(state_changes) + "}"
            state_ID = f"s{STATE_INDEX}"
            STATE_INDEX += 1
            PORT_INDEX += 1
            condition = "\mathbf{" + expr_to_penrose(condition)+ "}"
            transitions += f'''
                    State {state_ID}
                    Label {state_ID} ${state_changes_str}$
                    Port {port_name}
                    HasPort({comp_name}, {port_name}, "{input_topic}")
                    HasCompState({comp_name}, {state_ID})
                    PortTransition {port_name}_{PORT_INDEX} := HasPortTransition({comp_name}, {port_name}, "{condition}", {state_ID})
                                '''
            
        elif t["type"] == "interval":
            transitionID = expr_to_PlusCal(t["interval"])
            condition = expr_to_PlusCal(t["condition"])

            outputs = ""
            for o in t["outputs"]:
                outport = o["publisher"]["topic"]
                if outport not in topics:
                    topics.add(outport)
                    outputs += f"                        {outport} := {outport} (+) msg;\n"
            
            if len(t["state_changes"]) > 0:
                state_changes_list = []
                for sc in t["state_changes"]:
                    variable  = expr_to_PlusCal(sc["variable"])
                    new_value =  expr_to_PlusCal(sc["new_value"])
                    state_changes_list.append(f"{variable} = {new_value}")
                    state_changes = "[" + ", ".join(state_changes_list) + "]"

            else:
                state_changes = ''

            triggerID = "HZ_"+expr_to_PlusCal(t["interval"])
            condition = "\mathbf{" + expr_to_penrose(condition) + "}"
            transitions += f'''
                    PeriodicTrigger {triggerID}
                    Label {triggerID} "{t["interval"]}"
                    Port {comp_name}_{outport}
                    HasPort({comp_name}, {comp_name}_{outport}, "{outport}")
                    PeriodicTransition {triggerID}_p := HasPeriodicTransition({comp_name}, {triggerID}, "{condition}", {comp_name}_{outport})
                                '''
    return transitions
def penrose(files):

    spec = ""
    comp_names = []
    topic_names = []
    for file in files:
        comp_spec, comp_name = compPenrose(file)
        spec += comp_spec
        comp_names.append(comp_name)

    for topic in topics:
        topic_spec = topicTLA(topic)
        if topic_spec:
            topic_names.append(topic.capitalize())
            #spec += topic_spec
    tla_out= f'''
------------------------------- MODULE {system} -------------------------------
EXTENDS Sequences, Integers, TLC, FiniteSets
CONSTANTS {", ".join(comp_names)}, {", ".join(topic_names)}, Data, NULL, MaxQueue

ASSUME NULL \\notin Data


\* helper functions
''' + "SeqOf(set, n) == UNION {[1..m -> set] : m \\in 0..n} \* generates all sequences no longer than n consisting of elements in set" + f'''
seq (+) elem == Append(seq, elem)

(*--fair algorithm polling
{topicVariables()}
{spec}
end algorithm; *)
====
'''
    
    #print(tla_out)
    prenrose_out = ""
    for c in comp_names:
        prenrose_out += f'Component {c} \nLabel {c} "{c}"\n'
    prenrose_out += spec

    prenrose_out = prenrose_out.replace('"TRUE"', '""')
    
    print(prenrose_out)

    with open(f'./results/{system}_prenrose.substance', 'w') as file:
        file.write(prenrose_out)


def topicTLA(topic_name):
    inports_spec = ""
    if topic_name not in subs:
        return ''
    
    for sub in subs[topic_name]:
        inports_spec += f"                        {sub} := {sub} (+) msg;\n"
    return f'''
    fair process {topic_name} \in {topic_name.capitalize()}
        
        begin 
            Write: 
                if {topic_name} /= <<>> then
                    msg := Head({topic_name});
                    {topic_name} := Tail({topic_name});
{inports_spec}
                end if;
        end process;
    '''



def compPenrose(file):

    jsonComp = json.load(open(file))
    compInstance = jsonComp["node"] 
    compType = compInstance.capitalize()
    


    
    spec = ""
    for state_var in jsonComp["potential_state_vars"]:
        variable = expr_to_PlusCal(state_var["qualified_name"])
        initial_value = expr_to_PlusCal(str(state_var["initial-value"]["literal"]))
        state_init_expr = "\mathbf{" + name_to_penrose(variable) + ":=" + initial_value + "}"
        spec += f'''State {compType}_{variable}
                    HasCompState({compType}, {compType}_{variable})
                    HasState({compType}, {compType}_{variable}, "[{state_init_expr}]")
                    '''
    
    spec += transitionsPensrose(jsonComp)
    
    return spec, compType

    

In [25]:
system = "autoware02"
desired_output = "cubic_splines_viz"
configuration = {"g_sim_mode" : "TRUE"}
files =  ["./results/autoware-02/driving_planner.lattice_trajectory_gen.json"]
topics = set()
subs = {}
tla_out = penrose(files)

Component Lattice_trajectory_gen 
Label Lattice_trajectory_gen "Lattice_trajectory_gen"
State Lattice_trajectory_gen_g_waypoint_set
                    HasCompState(Lattice_trajectory_gen, Lattice_trajectory_gen_g_waypoint_set)
                    HasState(Lattice_trajectory_gen, Lattice_trajectory_gen_g_waypoint_set, "[\mathbf{g\_waypoint\_set:=False}]")
                    State Lattice_trajectory_gen_g_pose_set
                    HasCompState(Lattice_trajectory_gen, Lattice_trajectory_gen_g_pose_set)
                    HasState(Lattice_trajectory_gen, Lattice_trajectory_gen_g_pose_set, "[\mathbf{g\_pose\_set:=False}]")
                    
                    State s1
                    Label s1 $\mathbf{g\_pose\_set := True}$
                    Port Lattice_trajectory_gen_odom_pose
                    HasPort(Lattice_trajectory_gen, Lattice_trajectory_gen_odom_pose, "odom_pose")
                    HasCompState(Lattice_trajectory_gen, s1)
                    PortTransition Latt

In [26]:
system = "autoware03"
desired_output = "closest_waypoint"
configuration = {}
files =  ["./results/autoware-03/driving_planner.velocity_set.json"]
topics = set()
subs = {}
tla_out = penrose(files)

Component Velocity_set 
Label Velocity_set "Velocity_set"
State Velocity_set_g_path_flag
                    HasCompState(Velocity_set, Velocity_set_g_path_flag)
                    HasState(Velocity_set, Velocity_set_g_path_flag, "[\mathbf{g\_path\_flag:=False}]")
                    State Velocity_set_g_pose_flag
                    HasCompState(Velocity_set, Velocity_set_g_pose_flag)
                    HasState(Velocity_set, Velocity_set_g_pose_flag, "[\mathbf{g\_pose\_flag:=False}]")
                    State Velocity_set_false_count
                    HasCompState(Velocity_set, Velocity_set_false_count)
                    HasState(Velocity_set, Velocity_set_false_count, "[\mathbf{false\_count:=0}]")
                    State Velocity_set_prev_detection
                    HasCompState(Velocity_set, Velocity_set_prev_detection)
                    HasState(Velocity_set, Velocity_set_prev_detection, "[\mathbf{prev\_detection:=-1}]")
                    
                    State 

In [27]:
system = "autoware10"
desired_output = "obj_label_marker"
assumed_inputs = {"points_raw"}
configuration = {}
files =  ["./results/autoware-10/cv_tracker.obj_reproj.json"]
topics = set()
subs = {}
penrose(files)

Component Obj_reproj 
Label Obj_reproj "Obj_reproj"
State Obj_reproj_isReady_ndt_pose
                    HasCompState(Obj_reproj, Obj_reproj_isReady_ndt_pose)
                    HasState(Obj_reproj, Obj_reproj_isReady_ndt_pose, "[\mathbf{isReady\_ndt\_pose:=False}]")
                    State Obj_reproj_isReady_obj_pos_xyz
                    HasCompState(Obj_reproj, Obj_reproj_isReady_obj_pos_xyz)
                    HasState(Obj_reproj, Obj_reproj_isReady_obj_pos_xyz, "[\mathbf{isReady\_obj\_pos\_xyz:=False}]")
                    State Obj_reproj_ready_
                    HasCompState(Obj_reproj, Obj_reproj_ready_)
                    HasState(Obj_reproj, Obj_reproj_ready_, "[\mathbf{ready\_:=False}]")
                    
                    State s7
                    Label s7 $\mathbf{isReady\_obj\_pos\_xyz := False /\ isReady\_ndt\_pose := False}$
                    Port Obj_reproj_current_pose
                    HasPort(Obj_reproj, Obj_reproj_current_pose, "current_pose")